# Unsupervised Visual Anomaly Discovery for Industrial Quality Inspection
## Comprehensive Pipeline Walkthrough
**Author:** Gayathri (Unsupervised Learning & Dimensionality Reduction Lead)

This notebook demonstrates the end-to-end unsupervised pipeline:
1. **Feature Ingestion & Standardization**
2. **Dimensionality Reduction (PCA & Explained Variance)**
3. **Unsupervised Clustering (K-Means, DBSCAN, HDBSCAN)**
4. **Manifold Learning & Projections (t-SNE Visualizations)**
5. **Quantitative Evaluation (Silhouette, Davies-Bouldin, Calinski-Harabasz)**

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt

# Ensure project root is accessible
import sys
sys.path.append("..")

from src.dimensionality_reduction import scale_features, apply_pca, compute_explained_variance, find_optimal_components
from src.clustering import run_kmeans, compute_elbow_inertias, run_dbscan, run_hdbscan, extract_anomaly_indices, get_cluster_summary
from src.visualization import compute_tsne_embedding, plot_tsne_clusters, plot_pca_scree, plot_elbow_curve
from src.evaluation import compute_clustering_metrics

print("All modules imported successfully!")

### 1. Load or Simulate 2048-Dimensional Deep Feature Embeddings
When Tanya completes ResNet50 extraction, we load `datasets/features.npy`. In the meantime, we simulate 150 normal samples (across 3 sub-patterns) and 15 anomalies.

In [ ]:
from sklearn.datasets import make_blobs

if os.path.exists("../datasets/features.npy"):
    features = np.load("../datasets/features.npy")
    print("Loaded real features:", features.shape)
else:
    np.random.seed(42)
    X_normal, _ = make_blobs(n_samples=150, centers=3, n_features=2048, cluster_std=3.0, random_state=42)
    outliers = [np.random.randn(2048) / np.linalg.norm(np.random.randn(2048)) * np.random.uniform(30, 50) for _ in range(15)]
    features = np.vstack([X_normal, np.array(outliers)])
    print(f"Generated synthetic embeddings: {features.shape[0]} samples x {features.shape[1]} dimensions")

### 2. Standardization & Principal Component Analysis (PCA)

In [ ]:
# Scale features
scaled_features, _ = scale_features(features)

# Apply PCA retaining 95% of total variance
reduced_features, pca_model = apply_pca(scaled_features, explained_variance_ratio=0.95)
ind_var, cum_var = compute_explained_variance(pca_model)

print(f"Reduced from {features.shape[1]} dims down to {reduced_features.shape[1]} principal components!")
print(f"Cumulative variance retained: {cum_var[-1]*100:.2f}%")

# Scree Plot
fig_scree = plot_pca_scree(ind_var, cum_var, threshold=0.95)
plt.show()

### 3. K-Means Clustering & Elbow Curve

In [ ]:
k_vals, inertias = compute_elbow_inertias(reduced_features, k_range=list(range(2, 9)))
fig_elbow = plot_elbow_curve(k_vals, inertias)
plt.show()

kmeans_labels, _ = run_kmeans(reduced_features, n_clusters=3)
km_metrics = compute_clustering_metrics(reduced_features, kmeans_labels)
print("K-Means Intrinsic Metrics:", km_metrics)

### 4. Density-Based Clustering & Anomaly Discovery (DBSCAN & HDBSCAN)

In [ ]:
# DBSCAN
dbscan_labels, _ = run_dbscan(reduced_features, eps=25.0, min_samples=5)
db_summary = get_cluster_summary(dbscan_labels)
db_metrics = compute_clustering_metrics(reduced_features, dbscan_labels, exclude_noise=True)
print(f"DBSCAN: Found {db_summary['num_clusters']} clusters and {db_summary['num_anomalies']} anomalies")
print("DBSCAN Metrics:", db_metrics)

# HDBSCAN
hdbscan_labels, probs, _ = run_hdbscan(reduced_features, min_cluster_size=5)
hdb_summary = get_cluster_summary(hdbscan_labels)
hdb_metrics = compute_clustering_metrics(reduced_features, hdbscan_labels, exclude_noise=True)
print(f"HDBSCAN: Found {hdb_summary['num_clusters']} clusters and {hdb_summary['num_anomalies']} anomalies")
print("HDBSCAN Metrics:", hdb_metrics)

### 5. 2D Manifold Visualizations (t-SNE)

In [ ]:
tsne_2d = compute_tsne_embedding(reduced_features, n_components=2, perplexity=30.0)

# Plot DBSCAN results with visual anomaly highlighting
fig_db = plot_tsne_clusters(tsne_2d, dbscan_labels, title="t-SNE: DBSCAN Anomaly Discovery")
plt.show()